In [ ]:
!git clone -q https://github.com/uzairlol/ELICIT-fyp.git /kaggle/working/elicit
%cd /kaggle/working/elicit
!pip install -qq -r requirements.txt
!cd /kaggle/working/elicit && git log -1

In [ ]:
!apt-get update -y -qq > /dev/null
!apt-get install -y -qq pciutils zstd > /dev/null
# vLLM (on a Kaggle GPU T4 x2 session). Requires an internet-enabled session.
!pip install -qq vllm
!nvidia-smi

In [ ]:
# Start a vLLM server in the background.
# GGUF Q4_K_M = the same ~9 GB weights as `qwen2.5:14b` in Ollama (drop-in
# behavioural equivalent), served on one T4 with KV cache fully in VRAM.
#
# For the ~2x faster dual-GPU path (GPTQ int4, also ~9 GB) instead run:
#   python -m vllm.entrypoints.openai.api_server \
#     --model Qwen/Qwen2.5-14B-Instruct-GPTQ-Int4 \
#     --served-model-name qwen2.5-14b --tensor-parallel-size 2 \
#     --max-model-len 8192 --max-num-seqs 32 --gpu-memory-utilization 0.90

import subprocess
import time
import urllib.request

cmd = [
    "python",
    "-m",
    "vllm.entrypoints.openai.api_server",
    "--model",
    "Qwen/Qwen2.5-14B-Instruct-GGUF:Qwen2.5-14B-Instruct-Q4_K_M.gguf",
    "--served-model-name",
    "qwen2.5-14b",
    "--max-model-len",
    "8192",
    "--max-num-seqs",
    "16",
    "--gpu-memory-utilization",
    "0.92",
    "--swap-space",
    "0",
]

with open("/kaggle/working/vllm.log", "wb") as log_handle:
    vllm_proc = subprocess.Popen(
        cmd,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
    )

print("Waiting for vLLM to serve /v1/models...")
for _ in range(300):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=3) as response:
            if response.status == 200:
                print("vLLM is ready.")
                break
    except Exception:
        time.sleep(2)
else:
    print("Timed out; check /kaggle/working/vllm.log")

In [ ]:
!nvidia-smi
!tail -5 /kaggle/working/vllm.log

In [ ]:
import sys

sys.path.append("/kaggle/working/elicit/src")
%cd /kaggle/working/elicit/src

# Run seed 2 (adjust seeds or rounds as needed). --model-name must match
# the --served-model-name used to start vLLM.
!python run_experiments.py \
    --scenario ldf \
    --enable-ldf \
    --enable-climate-shocks \
    --model-name qwen2.5-14b \
    --seeds 2 \
    --num-rounds 30

In [ ]:
!tar -czvf /kaggle/working/simulation_results.tar.gz /kaggle/working/elicit/results/ /kaggle/working/elicit/data/